# Cuatro tareas, cuatro formas de leer un texto

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

| | |
|---|---|
| **Problema** | Un mismo texto puede necesitar una etiqueta, entidades, una respuesta o una traducción. |
| **Producto** | Cuatro exploraciones comparadas y una decisión de uso. |
| **Recursos** | Cuatro modelos, unos 1,7 GB de descargas. CPU suficiente. |

Un equipo recibe comentarios de vecinos, avisos de actividades y documentos de reglamento. Antes de programar nada tiene que decidir qué tarea resuelve cada necesidad.

**¿Qué tarea necesita cada problema y qué deja afuera el modelo que la resuelve?**

> **Cómo se trabaja.** Primero explorás cada tema con el código ya escrito y las notas **Para mirar**. Al final de cada bloque hay un **microlaboratorio**: una tabla te dice qué líneas completar y con qué, y la celda siguiente comprueba si quedó bien.

In [ ]:
# Importamos la API principal de Hugging Face para crear pipelines de NLP.
# `utils` trae helpers del curso: cargar datos y mostrar tablas.
from transformers import pipeline

import utils

utils.versiones()


## 1. Cuando las palabras no alcanzan

Antes de usar un modelo, probamos la herramienta más simple: contar cuántas palabras comparten una consulta y un aviso. Dos consultas, tres avisos.

In [ ]:
# Elegimos avisos y consultas de prueba para comparar coincidencias entre textos.
# Los índices corresponden a entradas de corpus.json y consultas.json.
corpus = utils.cargar_json("corpus.json")
consultas = utils.cargar_json("consultas.json")

avisos = [corpus[2]["texto"], corpus[7]["texto"], corpus[12]["texto"]]
# Posiciones: educacion_q7 (con palabra en común) y educacion_q2 (sin ella).
consultas_de_prueba = [consultas[7]["consulta"], consultas[2]["consulta"]]


### 1. Comparar palabras clave sin duplicados

Para comparar una consulta con un aviso, primero normalizamos el texto: lo convertimos a minúsculas y lo separamos en palabras. Luego usamos `set()` para eliminar duplicados y comparar solo la presencia de cada palabra. Así, una frase como "aprendo programación y programación" se trata como un conjunto de palabras únicas.

La intersección `A & B` devuelve las palabras que están en ambos textos. Ese criterio es útil para explorar coincidencias, aunque no reemplaza la comprensión semántica de un modelo.


In [ ]:
def contar_palabras_en_comun(consulta: str, texto: str) -> int:
    """Cuenta las palabras de más de tres letras presentes en la consulta y en el texto.

    Args:
        consulta: texto ingresado por la persona.
        texto: aviso o documento a comparar.

    Returns:
        Cantidad de palabras clave compartidas.
    """
    # Normalizamos y separamos en palabras; set() elimina duplicados.
    palabras_de_consulta = set(consulta.lower().split())
    # Quitamos puntos para que "biblioteca." y "biblioteca" se consideren iguales.
    palabras_de_texto = set(texto.lower().replace(".", "").split())

    cantidad = 0
    # La intersección contiene solo palabras presentes en ambos textos.
    for palabra in palabras_de_consulta & palabras_de_texto:
        if len(palabra) > 3:
            cantidad += 1
    return cantidad


filas = []
for consulta in consultas_de_prueba:
    for aviso in avisos:
        filas.append({
            "consulta": consulta,
            "aviso": aviso,
            "palabras_en_comun": contar_palabras_en_comun(consulta, aviso),
        })
utils.tabla(filas)


> **Para mirar.**
>
> - «quiero aprender fotografía» encuentra el taller porque comparte `fotografía`.
> - «necesito ayuda con las pruebas» no comparte ninguna palabra con las tutorías para «preparar los exámenes», que son su respuesta.
> - La coincidencia literal castiga la paráfrasis. Un modelo preentrenado ya aprendió relaciones entre palabras que no están escritas en el aviso.

## 2. Una tarea, una tubería

Un **`pipeline`** es una tubería de Hugging Face que junta tres pasos: preparar el texto, pasarlo por un modelo y devolver el resultado legible. Se le indican dos cosas: la **tarea** (`task`), que fija qué tipo de salida se espera, y el **modelo** (`model`), que es el identificador de un modelo ya entrenado. Cada tarea usa aquí un modelo **distinto** y devuelve una salida diferente. El nombre del modelo no garantiza que sirva para cualquier tarea o variedad del español.

Empezamos por la **clasificación de sentimiento**: asignar a cada texto una etiqueta (`POS`, `NEG` o `NEU`) y un puntaje.

In [ ]:
# Clasificación de sentimiento: a cada texto se le asigna una etiqueta
# (por ejemplo, POS, NEG o NEU) y un puntaje que mide la confianza.
clasificador = pipeline(
    "text-classification",
    model="pysentimiento/robertuito-sentiment-analysis",
)


In [ ]:
comentarios = [
    "Me encantó la atención en la biblioteca, volvería sin dudar.",
    "Qué lindo que el colectivo llegue siempre tarde, ¿no?",
    "La biblioteca abre de lunes a viernes.",
    "Otra vez sin turnos en el hospital, una vergüenza.",
]

resultados = utils.tabla(clasificador(comentarios))
resultados.insert(0, "texto", comentarios)
resultados.round(3)

> **Para mirar.**
>
> - El puntaje (`score`) no es la probabilidad de acertar: mide cuánto se inclina el modelo por esa etiqueta frente a las otras dos.
> - La ironía del segundo comentario salió `POS` (`0.39`): el modelo lee las palabras, no la intención. El aviso neutro salió `NEU` (`0.829`); la etiqueta (`label`) es siempre una de las tres.

## 3. Marcar partes del texto

La **clasificación de tokens** (o **reconocimiento de entidades nombradas**, NER) no etiqueta el texto entero: marca fragmentos y les asigna un tipo, como persona, lugar u organización. El parámetro `aggregation_strategy="simple"` une los pedazos consecutivos que el modelo etiquetó igual, para que un nombre de varias palabras salga como una sola entidad.

El aviso es sintético: organizaciones, lugares y una persona ficticios.

In [ ]:
# Reconocimiento de entidades nombradas: marcamos fragmentos del texto
# y les asignamos categorías como persona, lugar o institución.
aviso = utils.cargar_texto("aviso_ner.txt")
etiquetador = pipeline(
    "token-classification",
    model="mrm8488/bert-spanish-cased-finetuned-ner",
    aggregation_strategy="simple",
)

utils.tabla(etiquetador(aviso)).round(3)


> **Para mirar.**
>
> - Cada fila es un fragmento: `entity_group` es su tipo, `word` el texto, `score` el puntaje y `start` y `end` las posiciones en el aviso. El modelo marcó personas (`PER`), lugares (`LOC`), organizaciones (`ORG`) y el ciclo como `MISC`. No marcó ninguna de las dos fechas: no tiene un tipo para eso.
> - «Anfiteatro Municipal de Santa Fe» salió en dos fragmentos: `Anfiteatro Municipal` y `Santa Fe`. Los límites de una entidad pueden quedar corridos.
> - Marca fragmentos; no entiende de qué trata el aviso.

## 4. Responder con evidencia

En la **pregunta-respuesta extractiva** el modelo no redacta: recibe una pregunta y un contexto, y devuelve el fragmento del contexto que mejor la responde, con un puntaje y las posiciones donde empieza (`start`) y termina (`end`). Probamos tres preguntas que el documento responde y una que no.

In [ ]:
# Pregunta-respuesta extractiva: el modelo devuelve el fragmento del contexto
# que mejor responde la pregunta, sin redactar una respuesta nueva.
contexto = utils.cargar_texto("politicas_biblioteca.md")
respondedor = pipeline(
    "question-answering",
    model="mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es",
)

preguntas = [
    "¿Cuántos libros se pueden llevar por préstamo?",
    "¿Cuántos días dura el préstamo?",
    "¿A qué hora abre la biblioteca los sábados?",
    "¿Cuánto sale la multa por devolver un libro tarde?",
]

filas = []
for pregunta in preguntas:
    fila = {"pregunta": pregunta}
    fila.update(respondedor(question=pregunta, context=contexto))
    filas.append(fila)
utils.tabla(filas).round(3)


> **Para mirar.**
>
> - Las tres preguntas que el documento responde devolvieron un fragmento del contexto: `tres libros`, `14` y `10 a 14 horas` (columna `answer`). Las posiciones `start` y `end` dicen dónde está en el texto.
> - El documento no menciona multas, y el modelo igual devolvió `sin hojas sueltas ni anotaciones`. Su puntaje (`0.007`) es el más bajo de la tabla: una pista, no un aviso de que no sabe.

**¿Qué cambió al preguntar por la multa?** La respuesta aún es un fragmento literal del documento, pero **no responde la pregunta**. Que el puntaje sea bajo no autoriza a inventar un umbral universal de rechazo.

### Microlaboratorio 1 — Una respuesta que no responde

**Tarea.** Hacé una pregunta que el reglamento **no** responde y leé lo que el modelo devuelve.  
**Producto.** Una pregunta, una salida y un veredicto.  
**Antes de empezar.** La tabla de preguntas de la sección 4.

| Línea | Qué tiene que quedar | Con qué |
|---|---|---|
| (1) `mi_pregunta` | algo ausente del documento | por ejemplo, el costo de perder un libro |
| (2) `mi_respuesta` | lo que extrae el modelo | `respondedor(question=mi_pregunta, context=contexto)` |
| (3) `mi_veredicto` | si realmente responde, y por qué | una oración que mencione una evidencia del documento |

**Comprobación.** Verifica consulta y decisión. La pertinencia requiere cotejar el documento.

> **Ayuda 1.** Leé `datos/politicas_biblioteca.md`: ¿habla de costos?  
> **Ayuda 2.** La sección 4 muestra cómo llamar a `respondedor` con dos argumentos por nombre.

In [ ]:
mi_pregunta = ""  # COMPLETAR (1): algo ausente del documento
mi_respuesta = None  # COMPLETAR (2): consultar al respondedor
mi_veredicto = ""  # COMPLETAR (3): confrontar con el documento
mi_respuesta

In [ ]:
utils.comprobar(
    bool(mi_pregunta.strip()),
    "revisa (1): escribe una pregunta que el documento no responda",
)
utils.comprobar(
    mi_respuesta is not None
    and mi_respuesta["answer"] in contexto,
    "revisa (2): el resultado debe extraer texto del contexto",
)
utils.comprobar(
    len(mi_veredicto.split()) >= 15,
    "revisa (3): explica por que el fragmento responde o no",
)

## 5. Transformar el texto

La **traducción** produce un texto nuevo a partir del original. Probamos cuatro oraciones: una formal, una coloquial, una con nombre propio y una con «che».

In [ ]:
# Traducción automática: genera una versión del texto en otro idioma
# manteniendo la idea general, aunque puede perder matices.
traductor = pipeline("translation", model="Helsinki-NLP/opus-mt-es-en")

oraciones = [
    "Le informamos que la biblioteca permanecerá cerrada el lunes por el feriado.",
    "Estoy re cansado, hoy no me da el cuerpo para salir.",
    "Ana Peralta presentó el ciclo en la Biblioteca Almafuerte de Rosario.",
    "Che ¿me prestás el libro que te dejé el otro día?",
]

traducciones = utils.tabla(traductor(oraciones))
traducciones.insert(0, "original", oraciones)
traducciones


> **Para mirar.**
>
> - Conserva el contenido general: la primera y la cuarta oración salen bien, y `Ana Peralta` queda intacta.
> - Pierde registro y modismos: «hoy no me da el cuerpo para salir» se vuelve `today you don't give me the body to go out`, y `Biblioteca Almafuerte` pasa a `Almafuerte Library`.
> - Traducir no es comprender: el modelo reemplaza palabras y frases, sin saber qué quiso decir quien habló.

### Microlaboratorio 2: Tu propio caso

**Tarea.** Elegir una tarea, probarla con un texto tuyo y decidir para qué sirve y para qué no.  
**Producto.** Entrada, salida y decisión.  
**Antes de empezar.** Secciones 2 a 5.

| Línea | Qué tiene que quedar | Con qué |
|---|---|---|
| (1) `mi_entrada` | un texto tuyo, de la vida real | un texto entre comillas |
| (2) `mi_resultado` | la salida del pipeline que elegiste | `clasificador(...)`, `etiquetador(...)`, `traductor(...)` o `respondedor(question=mi_entrada, context=contexto)` |
| (3) `mi_decision` | una oración sobre para qué sirve y para qué no | al menos 15 palabras |

**Comprobación.** Revisa que haya entrada, salida y decisión; no si el modelo acertó.

> **Ayuda 1.** Las secciones 2 a 5 muestran cómo se llama a cada pipeline con un texto.  
> **Ayuda 2.** El resultado se guarda sin tocarlo: `mi_resultado = clasificador(mi_entrada)`.

In [ ]:
mi_entrada = ""  # COMPLETAR (1): un texto tuyo
mi_resultado = None  # COMPLETAR (2): llamar al pipeline elegido
mi_decision = ""  # COMPLETAR (3): para que sirve y para que no
mi_resultado

In [ ]:
utils.comprobar(
    len(mi_entrada.split()) > 0,
    "revisa (1): escribi un texto tuyo entre comillas",
)
utils.comprobar(
    mi_resultado is not None,
    "revisa (2): llama a un pipeline con mi_entrada",
)
utils.comprobar(
    len(mi_decision.split()) >= 15,
    "revisa (3): la decision tiene que tener al menos 15 palabras",
)

### Antes de elegir un modelo del Hub

Un identificador del Hub apunta a un repositorio; `pipeline` descarga el **checkpoint** y su tokenizador. En la ficha del Hub buscá primero: **qué tarea resuelve, para qué lenguas y datos fue preparado, qué licencia declara y qué límites reconoce**. La ficha es pública.

| Modelo de este cuaderno | Ficha para consultar |
|---|---|
| Sentimiento | [RoBERTuito](https://huggingface.co/pysentimiento/robertuito-sentiment-analysis) |
| Entidades | [BERT-NER](https://huggingface.co/mrm8488/bert-spanish-cased-finetuned-ner) |
| Respuestas | [BERT-QA](https://huggingface.co/mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es) |
| Traducción | [Helsinki-NLP es→en](https://huggingface.co/Helsinki-NLP/opus-mt-es-en) |

**Para investigar.** Elegí la ficha de la tarea que más te interesa: anotá un dato verificable de su preparación y un dato que **no encontraste**. Si no encontrás licencia o corpus, registrá esa ausencia.

## 6. Elegir con criterio

Cada tarea responde una pregunta distinta y tiene un límite propio. La decisión no es cuál es el mejor modelo, sino qué tarea necesita el problema. Confrontá la salida con el texto y con la ficha del modelo.

| Necesidad | Tarea | Salida | Límite |
|---|---|---|---|
| saber si un comentario es positivo | clasificación | etiqueta y puntaje | ironía y contexto |
| encontrar nombres y lugares | entidades | fragmentos | límites y categorías |
| responder desde un documento | QA extractiva | fragmento | solo lo que está en el contexto |
| entender un texto en otro idioma | traducción | texto nuevo | registro y cultura |

## Qué te llevás

- Un `pipeline` junta tarea y modelo: la tarea fija la forma de la salida; el modelo, su calidad.
- Cada salida es una decisión del modelo, no una verdad: hay que leerla con su límite.
- Devolver algo no prueba que sea adecuado.

**Sigue:** adaptar una tarea propia (cuaderno 02).